In [9]:
# 从纯文本导入 英文 分节经文 到数据库

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

def parse_filename(filename):
    name = os.path.splitext(filename)[0].replace("_en", "")
    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = get_book_id(book_abbr)
    return book_abbr, chapter, book_id


def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_en FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            print(f"➕ 新增 {verse_id}")

        elif row[0] == text_en:
            pass

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_en = ? WHERE id = ?",
                    (text_en, verse_id)
                )
                print(f"♻️  已覆盖 {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 导入完成")


# 只问文件名
if __name__ == "__main__":
    filename = input("请输入文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入文件名（如 Mt.1.txt）：Rom_6_en.txt
➕ 新增 Rom.6.1
➕ 新增 Rom.6.2
➕ 新增 Rom.6.3
➕ 新增 Rom.6.4
➕ 新增 Rom.6.5
➕ 新增 Rom.6.6
➕ 新增 Rom.6.7
➕ 新增 Rom.6.8
➕ 新增 Rom.6.9
➕ 新增 Rom.6.10
➕ 新增 Rom.6.11
➕ 新增 Rom.6.12
➕ 新增 Rom.6.13
➕ 新增 Rom.6.14
➕ 新增 Rom.6.15
➕ 新增 Rom.6.16
➕ 新增 Rom.6.17
➕ 新增 Rom.6.18
➕ 新增 Rom.6.19
➕ 新增 Rom.6.20
➕ 新增 Rom.6.21
➕ 新增 Rom.6.22
➕ 新增 Rom.6.23

✅ Rom.6 导入完成


In [11]:
# import_cn.py
# 从纯文本导入中文分节经文（text_cn）
# 文件名规范：2K_4_cn.txt

import sqlite3
import os
import re

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ==========================
# 从 book 表获取 book_id
# ==========================
def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

# ==========================
# 解析文件名（2K_4_cn.txt）
# ==========================
def parse_filename(filename):
    name = os.path.splitext(filename)[0]          # 去掉 .txt
    name = name.replace("_cn", "")                 # 去掉 _cn

    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)   # 2K
    chapter = int(match.group(2))  # 4
    book_id = get_book_id(book_abbr)

    return book_abbr, chapter, book_id

# ==========================
# 导入中文经文
# ==========================
def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?",
            (verse_id,)
        )
        row = cur.fetchone()

        # verse 不存在（极少见）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            print(f"➕ 新增 {verse_id}")

        elif row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            print(f"➕ 写入 text_cn: {verse_id}")

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖中文译文？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_cn = ? WHERE id = ?",
                    (text_cn, verse_id)
                )
                print(f"♻️  已覆盖 text_cn: {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 中文经文导入完成")

# ==========================
# 主程序
# ==========================
if __name__ == "__main__":
    filename = input("请输入中文经文文件名（如 2K_4_cn.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入中文经文文件名（如 2K_4_cn.txt）：Mt_10_cn.txt
➕ 写入 text_cn: Mt.10.1
➕ 写入 text_cn: Mt.10.2
➕ 写入 text_cn: Mt.10.3
➕ 写入 text_cn: Mt.10.4
➕ 写入 text_cn: Mt.10.5
➕ 写入 text_cn: Mt.10.6
➕ 写入 text_cn: Mt.10.7
➕ 写入 text_cn: Mt.10.8
➕ 写入 text_cn: Mt.10.9
➕ 写入 text_cn: Mt.10.10
➕ 写入 text_cn: Mt.10.11
➕ 写入 text_cn: Mt.10.12
➕ 写入 text_cn: Mt.10.13
➕ 写入 text_cn: Mt.10.14
➕ 写入 text_cn: Mt.10.15
➕ 写入 text_cn: Mt.10.16
➕ 写入 text_cn: Mt.10.17
➕ 写入 text_cn: Mt.10.18
➕ 写入 text_cn: Mt.10.19
➕ 写入 text_cn: Mt.10.20
➕ 写入 text_cn: Mt.10.21
➕ 写入 text_cn: Mt.10.22
➕ 写入 text_cn: Mt.10.23
➕ 写入 text_cn: Mt.10.24
➕ 写入 text_cn: Mt.10.25
➕ 写入 text_cn: Mt.10.26
➕ 写入 text_cn: Mt.10.27
➕ 写入 text_cn: Mt.10.28
➕ 写入 text_cn: Mt.10.29
➕ 写入 text_cn: Mt.10.30
➕ 写入 text_cn: Mt.10.31
➕ 写入 text_cn: Mt.10.32
➕ 写入 text_cn: Mt.10.33
➕ 写入 text_cn: Mt.10.34
➕ 写入 text_cn: Mt.10.35
➕ 写入 text_cn: Mt.10.36
➕ 写入 text_cn: Mt.10.37
➕ 写入 text_cn: Mt.10.38
➕ 写入 text_cn: Mt.10.39
➕ 写入 text_cn: Mt.10.40
➕ 写入 text_cn: Mt.10.41
➕ 写入 text_cn: Mt.10.42

✅ M

✅ verse 表数据已全部清空
